In [121]:
from datasets import load_dataset
import numpy as np
from collections import Counter
import os

# temporary use of
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset

In [6]:
ds = load_dataset("afmck/text8") # https://huggingface.co/datasets/afmck/text8

## The Skip-gram Model
the objective of the Skip-gram model is to maximize the average log probability

$$
\frac{1}{T} \sum_{t=1}^{T} \sum_{\substack{-c \le j \le c,j \ne 0}} \log p(w_{t+j} \mid w_t)
$$
- $c$ is the size of the training context (which can be a function of the center word $w_{t}$)

### Softmax

The basic Skip-gram formulation defines $p(w_{t+j} \mid w_t)$ using the softmax function:

$$
p(w_O \mid w_I) = \frac{\exp\left({v'_{w_O}}^{\top} v_{w_I}\right)}{\sum_{w=1}^{W} \exp\left({v'_w}^{\top} v_{w_I}\right)}
$$
- $v_{w}$ and $v'_{w}$ are the “input” and “output” vector representations of $w$, and $W$ is the number of words in the vocabulary.<br>
- This formulation is impractical because the cost of computing $\nabla \log p(w_O \mid w_I)$ is proportional to $W$, which is often very large ($10^5$–$10^7$ terms).

### Hierarchical Softmax

Then the hierarchical softmax defines $p(w_O \mid w_I)$ as follows:

$$
p(w \mid w_I) = \prod_{j=1}^{L(w)-1} \sigma \left( [\!\left[ n(w, j+1) = ch(n(w, j)) \right]\!\right] \cdot {v'}_{n(w,j)}^{\top} v_{w_I} )
$$

- Let $n(w, j)$ be the $j$-th node on the path from the root to $w$
- let $L(w)$ be the length of this path, so $n(w, 1) = root$ and $n(w, L(w)) = w$
- $\sigma(x) = {1} / {(1 + \exp(-x))}$. It can be verified that $\sum_{w=1}^{W} p(w \mid w_I) = 1$
- let $ch(n)$ be an arbitrary fixed child of $n$
- let $ \left[\!\left[ x \right]\!\right] $ be 1 if $x$ is true and -1 otherwise

### Negative Sampling

An alternative to the hierarchical softmax is Noise Contrastive Estimation (NCE)

We define Negative sampling (NEG) by the objective 

$$
\log \sigma\left({v'}_{w_O}^{\top} v_{w_I}\right) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)} \left[ \log \sigma\left(-{v'}_{w_i}^{\top} v_{w_I}\right) \right]
$$

which is used to replace every $\log P (w_O | w_I )$ term in the Skip-gram objective

- values of k in the range 5–20 are useful for small training datasets
- for large datasets the k can be as small as 2–5

> The main difference between the Negative sampling and NCE is that NCE needs both samples and the numerical probabilities of the noise distribution, while Negative sampling uses only samples

#### Noise Distribution

Both NCE and NEG have the noise distribution $P_n(w)$ as a free parameter

- unigram distribution $U(w)$ raised to the 3/4rd power (i.e., $U(w)^{3/4}/Z$) outperformed significantly the unigram and the uniform distributions

### Subsampling of Frequent Words

To counter the imbalance between the rare and frequent words, we used a simple subsampling approach: each word $w_i$ in the training set is discarded with probability computed by the formula

$$
P(w_i) = 1 - \sqrt{ \frac{t}{f(w_i)} }
$$

- $f(w_i)$ is the frequency of word $w_i$
- $t$ is a chosen threshold, typically around $10^{-5}$

We chose this subsampling formula because it aggressively subsamples words whose frequency is greater than $t$ while preserving the ranking of the frequencies

In [14]:
def split_data(ds):
    train, test, validation = ds["train"], ds["test"], ds["validation"]
    split_text = lambda dataset: [x for x in dataset["text"][0].split() if x]
    train = split_text(train)
    test = split_text(test)
    validation = split_text(validation)
    return train, test, validation

train, test, validation = split_data(ds)

In [15]:
def subsampling(words, threshold=1e-5):
    counts = Counter(words)
    total_count = len(words)
    # P(w) = (sqrt(f(w)/t) + 1) * (t/f(w))
    new_words = []
    for w in words:
        freq = counts[w] / total_count
        p_keep = (np.sqrt(freq / threshold) + 1) * (threshold / freq)
        if np.random.random() < p_keep:
            new_words.append(w)
    return new_words

In [16]:
def create_unigram_table(word_counts, vocab, table_size=100_000_000):
    freqs = np.array([word_counts[w] for w in vocab])
    pow_freqs = freqs**0.75
    probs = pow_freqs / sum(pow_freqs)
    
    # Tabloyu indekslerle doldur
    table = np.zeros(table_size, dtype=np.int32)
    count = 0
    for idx, p in enumerate(probs):
        n = int(p * table_size)
        table[count : count + n] = idx
        count += n
    return table
# neg_samples = table[np.random.randint(0, table_size, size=k)]

In [39]:
def set_seed(seed=42):
    # Python'ın kendi random motoru
    random.seed(seed)
    # OS seviyesinde (bazı kütüphane işlemleri için)
    os.environ['PYTHONHASHSEED'] = str(seed)
    # NumPy motoru
    np.random.seed(seed)
    # PyTorch CPU motoru
    torch.manual_seed(seed)
    
    # Eğer GPU (CUDA) kullanıyorsan
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Multi-GPU için
        
    # Apple Silicon (MPS) kullanıyorsan
    if torch.backends.mps.is_available():
        torch.manual_seed(seed)

    # İşlemleri deterministik (kesin) hale getir
    # Not: Bu işlemler eğitimi %5-10 yavaşlatabilir ama sonuçlar hep aynı çıkar
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Eğitime başlamadan önce çağır
set_seed(42) # Klasik 42, istersen başka sayı yap

In [93]:
# Frequency of the words
full_counts = Counter(train)
# Discard words occured less than 5 times & subsamble
train = [w for w in train if full_counts[w] >= 5]
train = subsampling(train)

# Create alphabetic vocabulary
vocab = sorted(list(set(train)))
# word to index & index to word operations
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab) # Vocabulary size

# Frequency of the words after discarding & subsampling
word_counts = {w: full_counts[w] for w in vocab}

# Unigram table for Negative Sampling
unigram_table = create_unigram_table(word_counts, vocab)

In [94]:
print(f'Sözlük Boyutu:        {vocab_size:,}')
print(f'Toplam kelime sayısı: {len(train):,}')

Sözlük Boyutu:        67,427
Toplam kelime sayısı: 1,932,350


In [122]:
class SkipGramDataset(IterableDataset):
    def __init__(self, data, word2idx, window_size=5):
        self.data = data
        self.word2idx = word2idx
        self.window_size = window_size

    def __iter__(self):
        for i, word in enumerate(self.data):
            target_idx = self.word2idx[word]

            window = np.random.randint(1, self.window_size + 1)

            start = max(0, i - window)
            end = min(len(self.data), i + window + 1)

            context_positions = [j for j in range(start, end) if j != i]

            if not context_positions:
                continue

            j = context_positions[np.random.randint(len(context_positions))]

            yield target_idx, self.word2idx[self.data[j]]

In [133]:
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.u_embeddings = nn.Embedding(vocab_size, emb_dim) # Center words
        self.v_embeddings = nn.Embedding(vocab_size, emb_dim) # Context & Negative words
        
        # Ağırlıkları başlatma
        init_range = 0.5 / emb_dim
        self.u_embeddings.weight.data.uniform_(-init_range, init_range)
        self.v_embeddings.weight.data.zero_()

    def forward(self, pos_u, pos_v, neg_v):
        # pos_u: [batch_size]
        # pos_v: [batch_size]
        # neg_v: [batch_size, k]

        emb_u = self.u_embeddings(pos_u)     # [batch, dim]
        emb_v = self.v_embeddings(pos_v)     # [batch, dim]
        emb_neg = self.v_embeddings(neg_v)   # [batch, k, dim]

        # Pozitif Skor: dot(u, v)
        # Her batch için tek tek nokta çarpımı
        pos_score = torch.sum(torch.mul(emb_u, emb_v), dim=1) 
        pos_score = torch.clamp(pos_score, max=10, min=-10) # Sayısal stabilite
        pos_loss = F.logsigmoid(pos_score)

        # Negatif Skor: u * neg_v^T
        # [batch, 1, dim] * [batch, dim, k] -> [batch, 1, k]
        neg_score = torch.bmm(emb_neg, emb_u.unsqueeze(2)).squeeze()
        neg_score = torch.clamp(neg_score, max=10, min=-10)
        neg_loss = torch.sum(F.logsigmoid(-neg_score), dim=1)

        return -(pos_loss + neg_loss).mean()

In [169]:
# --- Hiperparametreler ---
EMB_DIM = 512
BATCH_SIZE = 1024
K_NEG = 5 
LEARNING_RATE = 0.001
EPOCHS = 15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Hazırlık
model = SkipGramModel(vocab_size, EMB_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE) # Word2Vec için Adam iyidir
dataset = SkipGramDataset(train, word2idx, window_size=5)
loader = DataLoader(dataset, batch_size=BATCH_SIZE)

In [171]:
print(f"Eğitim başlıyor... Cihaz: {DEVICE}")

EPOCHS = 10000
for epoch in range(EPOCHS):
    total_loss = 0
    for i, (pos_u, pos_v) in enumerate(loader):
        # 1. Negatif örnekleri tablodan çek (Batch bazlı)
        # unigram_table: Daha önce oluşturduğumuz NumPy array
        neg_v = np.random.choice(unigram_table, size=(pos_u.size(0), K_NEG))
        
        # Verileri cihaza taşı
        pos_u = pos_u.to(DEVICE)
        pos_v = pos_v.to(DEVICE)
        neg_v = torch.from_numpy(neg_v).long().to(DEVICE)

        # 2. Forward & Loss
        loss = model(pos_u, pos_v, neg_v)

        # 3. Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Ara bilgilendirme
        if i % 1000 == 0:
            print(f"Epoch: {epoch}, Batch: {i}, Loss: {loss.item():.4f}")

    print(f"--- Epoch {epoch} bitti. Ortalama Loss: {total_loss/i:.4f} ---")

Eğitim başlıyor... Cihaz: mps
Epoch: 0, Batch: 0, Loss: 2.9201


KeyboardInterrupt: 

In [ ]:
def vector(word):
    if word not in word2idx:
        return None
    return wv[word2idx[word]]

def find_closest_word(target_vector, top_k=5, exclude_words=[]):
    similarities = np.dot(wv, target_vector)
    sorted_indices = np.argsort(similarities)[::-1]
    
    results = []
    for idx in sorted_indices:
        word = idx2word[idx]
        if word not in exclude_words:
            return word, similarities[idx]
            results.append((word, similarities[idx]))
        if len(results) >= top_k:
            break
    return results

In [ ]:
# Vektörleri modelden çek (Merkez vektörleri u_embeddings kullanılır)
word_vectors = model.u_embeddings.weight.data.cpu()

# Her vektörü kendi uzunluğuna böl (Normalization)
# Bu sayede nokta çarpımı doğrudan Cosine Similarity olur
norms = word_vectors.norm(p=2, dim=1, keepdim=True)
word_vectors = word_vectors / norms

# Kolay erişim için NumPy'a çevirebilirsin
wv = word_vectors.numpy()

In [ ]:
king = vector("king")
man = vector("man")
woman = vector("woman")
queen = find_closest_word(king - man + woman)
queen